In [28]:
%env CUDA_LAUNCH_BLOCKING=1

env: CUDA_LAUNCH_BLOCKING=1


# --- 1. Load and preprocess data ---

In [29]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

playlist = pd.read_csv("playlist_with_segment.csv", engine='python', on_bad_lines='skip') # Because the file may have some parsing errors
tracks = pd.read_csv("/content/tracks_new.csv")

In [30]:
playlist = playlist.head(1000)
playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

<ipython-input-30-b51a2be1e7e9>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
<ipython-input-30-b51a2be1e7e9>:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))


In [31]:
playlist_sentiment_df = playlist['sentiment_centroid'].apply(pd.Series)
playlist_sentiment_df.columns = [f'sent{i+1}' for i in range(playlist_sentiment_df.shape[1])]
playlist = pd.concat([playlist.drop(columns=['sentiment_centroid']), playlist_sentiment_df], axis=1)

In [32]:
playlist_genre_df = playlist['genre_centroid'].apply(pd.Series)
playlist_genre_df.columns = [f'genre{i+1}' for i in range(playlist_genre_df.shape[1])]
playlist = pd.concat([playlist.drop(columns=['genre_centroid']), playlist_genre_df], axis=1)

In [33]:
playlist_relevant_columns = playlist[['playlist_idx', 'track_idx_list', 'tracks_to_predict',
                                      'popularity_mean',
                                      'era_early_years_proportion', 'era_classic_era_proportion',
                                      'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion',
                                      'length_short_proportion', 'length_medium_proportion', 'length_long_proportion',
                                      'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
                                      'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']]

tracks_relevant_columns = tracks[['track_idx', 'track_popularity',
                                  'Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era',
                                  'Short', 'Medium', 'Long',
                                  "joy", "calm", "sadness", "fear", "energizing", "dreamy",
                                  "Instrumental / Ambient Sounds", "Soft Acoustic / Classical", "Orchestral / Soundtrack",
                                  "Mid-tempo Pop / Indie", "Upbeat Electronic / Dance", "Slow & Melancholic (Sad Songs)",
                                  "Experimental / Jazz Fusion", "Lo-Fi / Chill Vibes"]]

playlist_relevant_columns['popularity_mean'] /= 100
tracks_relevant_columns['track_popularity'] /= 100

<ipython-input-33-32399866e1bd>:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['popularity_mean'] /= 100
<ipython-input-33-32399866e1bd>:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tracks_relevant_columns['track_popularity'] /= 100


In [34]:
def convert_string_array_to_list(s):
    if isinstance(s, str):
        numbers = re.findall(r'\d+', s)
        return [int(x) for x in numbers]
    elif isinstance(s, np.ndarray):
        return s.astype(int).tolist()
    elif isinstance(s, list):
        return [int(x) for x in s if str(x).isdigit()]
    return []

playlist_relevant_columns['track_idx_list'] = playlist_relevant_columns['track_idx_list'].apply(convert_string_array_to_list)
playlist_relevant_columns['tracks_to_predict'] = playlist_relevant_columns['tracks_to_predict'].apply(convert_string_array_to_list)

<ipython-input-34-84f4ca8a0bf9>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['track_idx_list'] = playlist_relevant_columns['track_idx_list'].apply(convert_string_array_to_list)
<ipython-input-34-84f4ca8a0bf9>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['tracks_to_predict'] = playlist_relevant_columns['tracks_to_predict'].apply(convert_string_array_to_list)


# --- 2. Standardize features ---

In [35]:
from sklearn.preprocessing import StandardScaler

popularity_cols = ['track_popularity']
playlist_popularity_cols = ['popularity_mean']

era_cols = ['Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era']
playlist_era_cols = ['era_early_years_proportion', 'era_classic_era_proportion',
                     'era_golden_era_proportion', 'era_2000s_proportion', 'era_modern_era_proportion']

length_cols = ['Short', 'Medium', 'Long']
playlist_length_cols = ['length_short_proportion', 'length_medium_proportion', 'length_long_proportion']

sentiment_cols = ['joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy']
playlist_sentiment_cols = ['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']

genre_cols = ['Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
              'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
              'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']
playlist_genre_cols = ['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']

In [36]:
scalers = {
    'popularity_playlist': StandardScaler(),
    'era_playlist': StandardScaler(),
    'length_playlist': StandardScaler(),
    'sentiment_playlist': StandardScaler(),
    'genre_playlist': StandardScaler(),
    'popularity_track': StandardScaler(),
    'era_track': StandardScaler(),
    'length_track': StandardScaler(),
    'sentiment_track': StandardScaler(),
    'genre_track': StandardScaler(),
}

scalers['popularity_playlist'].fit(playlist_relevant_columns[playlist_popularity_cols])
scalers['era_playlist'].fit(playlist_relevant_columns[playlist_era_cols])
scalers['length_playlist'].fit(playlist_relevant_columns[playlist_length_cols])
scalers['sentiment_playlist'].fit(playlist_relevant_columns[playlist_sentiment_cols])
scalers['genre_playlist'].fit(playlist_relevant_columns[playlist_genre_cols])

scalers['popularity_track'].fit(tracks_relevant_columns[popularity_cols])
scalers['era_track'].fit(tracks_relevant_columns[era_cols])
scalers['length_track'].fit(tracks_relevant_columns[length_cols])
scalers['sentiment_track'].fit(tracks_relevant_columns[sentiment_cols])
scalers['genre_track'].fit(tracks_relevant_columns[genre_cols])

StandardScaler()

In [37]:
playlist_relevant_columns[playlist_popularity_cols] = scalers['popularity_playlist'].transform(playlist_relevant_columns[playlist_popularity_cols])
tracks_relevant_columns[popularity_cols] = scalers['popularity_track'].transform(tracks_relevant_columns[popularity_cols])

playlist_relevant_columns[playlist_era_cols] = scalers['era_playlist'].transform(playlist_relevant_columns[playlist_era_cols])
tracks_relevant_columns[era_cols] = scalers['era_track'].transform(tracks_relevant_columns[era_cols])

playlist_relevant_columns[playlist_length_cols] = scalers['length_playlist'].transform(playlist_relevant_columns[playlist_length_cols])
tracks_relevant_columns[length_cols] = scalers['length_track'].transform(tracks_relevant_columns[length_cols])

playlist_relevant_columns[playlist_sentiment_cols] = scalers['sentiment_playlist'].transform(playlist_relevant_columns[playlist_sentiment_cols])
tracks_relevant_columns[sentiment_cols] = scalers['sentiment_track'].transform(tracks_relevant_columns[sentiment_cols])

playlist_relevant_columns[playlist_genre_cols] = scalers['genre_playlist'].transform(playlist_relevant_columns[playlist_genre_cols])
tracks_relevant_columns[genre_cols] = scalers['genre_track'].transform(tracks_relevant_columns[genre_cols])


<ipython-input-37-9bf352533320>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns[playlist_popularity_cols] = scalers['popularity_playlist'].transform(playlist_relevant_columns[playlist_popularity_cols])
<ipython-input-37-9bf352533320>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tracks_relevant_columns[popularity_cols] = scalers['popularity_track'].transform(tracks_relevant_columns[popularity_cols])
<ipython-input-37-9bf352533320>:4: SettingWithCopyWarning: 
A value is trying t

# --- 3. Prepare data for DNN ---

In [38]:
playlist_features = playlist_relevant_columns[
    playlist_popularity_cols + playlist_era_cols + playlist_length_cols +
    playlist_sentiment_cols + playlist_genre_cols
].values

track_features = tracks_relevant_columns[
    popularity_cols + era_cols + length_cols + sentiment_cols + genre_cols
].values

playlist_idx_to_features = {
    playlist_relevant_columns.iloc[i]['playlist_idx']: playlist_features[i]
    for i in range(len(playlist_relevant_columns))
}

track_idx_to_features = {
    tracks_relevant_columns.iloc[i]['track_idx']: track_features[i]
    for i in range(len(tracks_relevant_columns))
}

# --- 4. DNN Model ---

## 🔧 **Step 1: Setup and Install Required Packages**

### 🔍 What it does:
- Import **PyTorch** packages.
- Detect and set the device to **GPU** (`cuda`) if available (which is the case in Colab), else default to CPU.

### ✅ Why:
Using a GPU will **significantly speed up training and inference** in deep learning models, especially when processing large numbers of playlist-track pairs.

In [39]:
# PyTorch for DNN
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 📦 **Step 2: Prepare Training Dataset**

### 🔍 What it does:
- Extracts and stores **feature vectors** for playlists and tracks into dictionaries.
- `playlist_idx_to_features`: maps each playlist index to its standardized feature vector.
- `track_idx_to_features`: maps each track index to its standardized feature vector.

### ✅ Why:
We need to prepare feature data in a way that makes it easy to generate all (playlist, track) pairs for training a supervised model.

In [40]:
# Combine all playlist and track features for modeling
playlist_features = playlist_relevant_columns[
    playlist_popularity_cols + playlist_era_cols + playlist_length_cols +
    playlist_sentiment_cols + playlist_genre_cols
].values

track_features = tracks_relevant_columns[
    popularity_cols + era_cols + length_cols + sentiment_cols + genre_cols
].values

playlist_idx_to_features = {
    playlist_relevant_columns.iloc[i]['playlist_idx']: playlist_features[i]
    for i in range(len(playlist_relevant_columns))
}

track_idx_to_features = {
    tracks_relevant_columns.iloc[i]['track_idx']: track_features[i]
    for i in range(len(tracks_relevant_columns))
}

## 📚 **Step 3: Define PyTorch Dataset**

### 🔍 What it does:
- Defines a custom `PlaylistTrackDataset` class that:
  - **Generates positive pairs** (playlist + track that it contains or should predict).
  - **Generates negative pairs** (random tracks not in that playlist).
- Each sample is a tuple: `([playlist_features ⨁ track_features], label)`.

### ✅ Why:
This dataset allows the model to **learn the difference between relevant and irrelevant tracks** for a given playlist using supervised learning. We treat this as a **binary classification problem**.

In [41]:
import random

class PlaylistTrackDataset(Dataset):
    def __init__(self, playlist_df, num_negative_samples=5):
        self.samples = []
        all_track_ids = list(track_idx_to_features.keys())

        for _, row in playlist_df.iterrows():
            pid = row['playlist_idx']
            pos_track_ids = row['tracks_to_predict']

            # Positive samples
            for tid in pos_track_ids:
                if tid in track_idx_to_features:
                    self.samples.append((pid, tid, 1))

            # Negative samples
            neg_candidates = list(set(all_track_ids) - set(pos_track_ids))
            neg_sampled = random.sample(neg_candidates, min(len(pos_track_ids) * num_negative_samples, len(neg_candidates)))

            for tid in neg_sampled:
                self.samples.append((pid, tid, 0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, tid, label = self.samples[idx]
        p_feat = torch.tensor(playlist_idx_to_features[pid], dtype=torch.float32)
        t_feat = torch.tensor(track_idx_to_features[tid], dtype=torch.float32)
        x = torch.cat([p_feat, t_feat])
        y = torch.tensor([float(label)], dtype=torch.float32)  # Force float32 in [0, 1]
        return x, y

## 🧠 **Step 4: Define the DNN Model**

### 🔍 What it does:
- Defines a simple **3-layer feedforward neural network** with:
  - ReLU activations
  - Dropout and BatchNorm for regularization and stability
  - A final Sigmoid layer to output a **score between 0 and 1**

### ✅ Why:
This model takes the concatenated playlist-track feature vector and learns **nonlinear patterns** that determine how "suitable" a track is for a playlist. Much more expressive than cosine similarity.

In [42]:
class DNNRecommender(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)  # No sigmoid here
        )
    def forward(self, x):
        return self.model(x).squeeze()

## ⚙️ **Step 5: Train the Model**

### 🔍 What it does:
- Constructs a `DataLoader` for mini-batch training.
- Trains the model for a few epochs using **Binary Cross-Entropy Loss**.
- Prints loss after each epoch.

### ✅ Why:
Training allows the model to learn which combinations of playlist and track features result in high similarity scores. The loss guides the model to separate good recommendations (label=1) from bad ones (label=0).

In [43]:
# Create dataset and dataloader
train_dataset = PlaylistTrackDataset(playlist_relevant_columns)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

# Instantiate model
input_size = len(playlist_features[0]) + len(track_features[0])
print("Input size:", input_size)
model = DNNRecommender(input_size).to(device)

# Optimizer and loss
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
# Count positive and negative labels
num_pos = sum(1 for _, _, label in train_dataset.samples if label == 1)
num_neg = sum(1 for _, _, label in train_dataset.samples if label == 0)
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Training loop
EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        preds = model(x_batch)
        preds = preds.view(-1)       # flatten
        y_batch = y_batch.view(-1)   # flatten

        if not torch.all(torch.isfinite(preds)):
          print("⚠️ preds contains NaNs or Infs!")
          print(preds)
          break

        if not torch.all(torch.isfinite(y_batch)):
            print("⚠️ y_batch contains NaNs or Infs!")
            print(y_batch)
            break

        if not ((y_batch >= 0).all() and (y_batch <= 1).all()):
            print("⚠️ y_batch contains values outside [0, 1]!")
            print(y_batch)
            break

        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss:.4f}")

Input size: 46
Epoch 1/20, Loss: 56.1387
Epoch 2/20, Loss: 51.5805
Epoch 3/20, Loss: 49.5195
Epoch 4/20, Loss: 48.6004
Epoch 5/20, Loss: 48.0638
Epoch 6/20, Loss: 46.0138
Epoch 7/20, Loss: 45.3775
Epoch 8/20, Loss: 44.5367
Epoch 9/20, Loss: 44.1865
Epoch 10/20, Loss: 43.7787
Epoch 11/20, Loss: 42.2126
Epoch 12/20, Loss: 41.8097
Epoch 13/20, Loss: 41.4092
Epoch 14/20, Loss: 40.9091
Epoch 15/20, Loss: 40.6400
Epoch 16/20, Loss: 39.6295
Epoch 17/20, Loss: 39.2422
Epoch 18/20, Loss: 38.9450
Epoch 19/20, Loss: 38.8183
Epoch 20/20, Loss: 38.6349


## 🎯 **Step 6: Generate Recommendations Using the Trained Model**

### 🔍 What it does:
- For each playlist:
  - Pairs it with **every track**.
  - Computes scores using the trained model.
  - **Removes tracks already seen** in the playlist.
  - Sorts by predicted score and selects the top 50 tracks.

### ✅ Why:
This step replaces cosine similarity. Now, your recommendations are powered by a model that **learns from real training signals** instead of assuming equal weights across features.

In [44]:
# Make predictions for all playlists and tracks
model.eval()
playlist_ids = playlist_relevant_columns['playlist_idx'].tolist()
track_ids = list(track_idx_to_features.keys())

recommendations = {}

with torch.no_grad():
    for pid in playlist_ids:
        p_feat = torch.tensor(playlist_idx_to_features[pid], dtype=torch.float32).to(device)
        p_feat_repeated = p_feat.repeat(len(track_ids), 1)

        # Fix: stack into one NumPy array first
        t_feats_np = np.array([track_idx_to_features[tid] for tid in track_ids], dtype=np.float32)
        t_feats = torch.from_numpy(t_feats_np).to(device)

        inputs = torch.cat([p_feat_repeated, t_feats], dim=1)

        scores = torch.sigmoid(model(inputs)).cpu().numpy()
        track_scores = list(zip(track_ids, scores))

        seen = set(map(int, playlist_relevant_columns.loc[playlist_relevant_columns['playlist_idx'] == pid, 'track_idx_list'].values[0]))
        filtered = [(tid, s) for tid, s in track_scores if tid not in seen]

        top_50 = sorted(filtered, key=lambda x: x[1], reverse=True)[:50]
        recommendations[pid] = [tid for tid, _ in top_50]

# Update recommendations in the DataFrame
playlist_relevant_columns['recommendations_dnn'] = playlist_relevant_columns['playlist_idx'].map(recommendations)

<ipython-input-44-e34302ba1f46>:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['recommendations_dnn'] = playlist_relevant_columns['playlist_idx'].map(recommendations)


# 📈 **Step 7: Evaluate the DNN Recommendations**

### 🔍 What it does:
- Reuses your existing evaluation logic (Hit Ratio, MRR, MAP@50).
- Applies it to the model’s new recommendations stored in `recommendations_dnn`.

### ✅ Why:
Same as before — but now we're evaluating the **learned recommendations** from the DNN. This gives us an apples-to-apples comparison with your original content-based filtering method.

In [56]:
def compute_metrics_for_playlist(predicted_tracks, test_indices, k):
    top_k = predicted_tracks[:k]  # Consider only top K predictions

    # Hit@K
    hit = int(any(t in top_k for t in test_indices))

    # MRR and AP calculations
    precisions = []
    num_hits = 0
    mrr = 0.0

    for rank_idx, track_idx in enumerate(top_k):
        if track_idx in test_indices:
            num_hits += 1
            precision_at_k = num_hits / (rank_idx + 1)
            precisions.append(precision_at_k)
            if mrr == 0.0:
                mrr = 1.0 / (rank_idx + 1)

    ap = np.mean(precisions) if precisions else 0.0

    return hit, mrr, ap

def evaluate_dnn_model(playlist_df, k):
    hit_total, mrr_total, ap_total = 0, 0, 0
    for _, row in playlist_df.iterrows():
        test_indices = row['tracks_to_predict']
        predicted_tracks = row['recommendations_dnn'][:k]

        hit, mrr, ap = compute_metrics_for_playlist(predicted_tracks, test_indices, k)
        hit_total += hit
        mrr_total += mrr
        ap_total += ap

    n = len(playlist_df)
    return hit_total/n, mrr_total/n, ap_total/n

# Run evaluation
hit, mrr, map_ = evaluate_dnn_model(playlist_relevant_columns, k=50)

print(f"Hit@50 (DNN): {hit:.4f}")
print(f"MRR (DNN): {mrr:.4f}")
print(f"MAP@50 (DNN): {map_:.4f}")

Hit@50 (DNN): 0.1220
MRR (DNN): 0.0317
MAP@50 (DNN): 0.0300


In [57]:
print(playlist_relevant_columns.head())

   playlist_idx                                     track_idx_list  \
0             1  [3689, 207774, 194775, 135193, 218011, 37844, ...   
1             2  [160375, 131195, 164629, 147280, 193891, 17077...   
2             3  [104436, 229428, 25968, 186871, 81592, 15947, ...   
3             4  [244173, 210510, 9349, 224202, 147251, 35498, ...   
4             5  [194410, 10513, 62267, 196463, 14164, 13405, 1...   

                                   tracks_to_predict  popularity_mean  \
0  [218708, 242974, 165272, 9418, 221860, 229224,...        -0.107828   
1  [232845, 111887, 144160, 10663, 216321, 138869...         0.756258   
2  [42986, 156208, 73950, 54114, 134077, 214967, ...         0.578014   
3  [166268, 22122, 211269, 71335, 21853, 190702, ...         0.066122   
4  [209088, 186942, 33032, 27612, 33426, 201531, ...         0.671317   

   era_early_years_proportion  era_classic_era_proportion  \
0                   -0.130619                   -0.316579   
1                 

In [59]:
# Evaluate DNN model by cluster
cluster_metrics = []

playlist_relevant_columns['cluster'] = playlist['cluster']
recommendation = playlist_relevant_columns[['playlist_idx', 'cluster', 'track_idx_list', 'tracks_to_predict', 'recommendations', 'tracks_to_predict_similarity']]
recommendation['recommendations_dnn'] = recommendation['playlist_idx'].map(recommendations)

for cluster_id, group in recommendation.groupby("cluster"):
    hit, mrr, map_ = evaluate_dnn_model(group, k=50)
    cluster_metrics.append({
        "Cluster": cluster_id,
        "Hit@50": round(hit, 4),
        "MRR": round(mrr, 4),
        "MAP@50": round(map_, 4)
    })

# Display results as a DataFrame
import pandas as pd
metrics_by_cluster_df = pd.DataFrame(cluster_metrics).sort_values("Cluster").reset_index(drop=True)
print(metrics_by_cluster_df)

<ipython-input-59-d8a00a3adba6>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  playlist_relevant_columns['cluster'] = playlist['cluster']
<ipython-input-59-d8a00a3adba6>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recommendation['recommendations_dnn'] = recommendation['playlist_idx'].map(recommendations)


   Cluster  Hit@50     MRR  MAP@50
0        0  0.0541  0.0073  0.0070
1        1  0.0954  0.0160  0.0150
2        2  0.2105  0.0827  0.0781
3        3  0.0950  0.0077  0.0076
